# Cotton benchmark — gold set v2 (teacher pre-annotations)
Re-predicts all test tiles with the trained teacher, filters impossible box
sizes, samples ~1,200 stratified gold tiles, packages for CVAT.
Needs in Drive `cotton_bench/`: `cotton_test_tiles.zip`, `teacher_11s_best.pt`,
`test_classes.csv`. Runtime: T4 GPU, ~15 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
!pip install -q ultralytics
!unzip -q {DRIVE}/cotton_test_tiles.zip -d /content/
import glob
tiles = sorted(set(glob.glob('/content/test_tiles/*.jpg') + glob.glob('/content/test_tiles/*.JPG')))
print(len(tiles), 'tiles')

In [ ]:
from ultralytics import YOLO
import os, json
from tqdm import tqdm

model = YOLO(f'{DRIVE}/teacher_11s_best.pt')
preds = {}
BATCH = 64
for i in tqdm(range(0, len(tiles), BATCH)):
    batch = tiles[i:i+BATCH]
    results = model.predict(batch, imgsz=640, conf=0.20, iou=0.45,
                            max_det=3000, verbose=False)
    for path, r in zip(batch, results):
        b = r.boxes
        preds[os.path.basename(path)] = [
            [float(x1), float(y1), float(x2), float(y2), float(c)]
            for (x1, y1, x2, y2), c in
            zip(b.xyxy.cpu().numpy(), b.conf.cpu().numpy())]

with open(f'{DRIVE}/teacher_test_predictions.json', 'w') as f:
    json.dump(preds, f)
print('saved teacher predictions for', len(preds), 'tiles')

In [ ]:
# Box size distribution -> size filter
import numpy as np
sides = np.array([max(x2-x1, y2-y1)
                  for boxes in preds.values()
                  for x1, y1, x2, y2, c in boxes])
print('max box side (px): median %.0f  p90 %.0f  p99 %.0f  p99.9 %.0f  max %.0f'
      % tuple(np.percentile(sides, [50, 90, 99, 99.9, 100])))

MAX_SIDE = 120   # ~38 cm on the ground; nothing boll-like is bigger
filtered = {t: [b for b in boxes if max(b[2]-b[0], b[3]-b[1]) <= MAX_SIDE]
            for t, boxes in preds.items()}
removed = sum(len(preds[t]) - len(filtered[t]) for t in preds)
print(f'size filter removed {removed} boxes '
      f'({removed / max(len(sides),1) * 100:.2f}%)')

In [ ]:
# Stratified gold sample v2: ~1200 tiles, ~600 per class
import re, csv, random
random.seed(43)

cls_of_stem = {}
with open(f'{DRIVE}/test_classes.csv') as f:
    for row in csv.DictReader(f):
        cls_of_stem[row['stem']] = row['class']

def stem_of(name): return re.split(r'_r\d+_c\d+_x\d+_y\d+', name)[0]
def dbin(n):
    return ('0' if n == 0 else '1-20' if n <= 20 else '21-60' if n <= 60
            else '61-120' if n <= 120 else '120+')

strata = {}
for t, boxes in filtered.items():
    strata.setdefault((cls_of_stem.get(stem_of(t), '?'), dbin(len(boxes))),
                      []).append(t)

PER_CLASS = 600
gold = []
print(f"{'class':<18}{'density':<10}{'avail':<8}{'sampled'}")
for cls in sorted({k[0] for k in strata}):
    cstr = {b: lst for (c, b), lst in strata.items() if c == cls}
    ctot = sum(len(v) for v in cstr.values())
    for b, lst in sorted(cstr.items()):
        n = min(len(lst), max(15, round(PER_CLASS * len(lst) / ctot)))
        gold.extend(random.sample(lst, n))
        print(f"{cls:<18}{b:<10}{len(lst):<8}{n}")
print('gold total:', len(gold))

In [ ]:
# Package for CVAT
import shutil
coco = {'images': [], 'annotations': [],
        'categories': [{'id': 1, 'name': 'cotton_boll'}]}
ann_id = 1
os.makedirs('/content/gold_v2', exist_ok=True)
for img_id, t in enumerate(sorted(gold), 1):
    shutil.copy(f'/content/test_tiles/{t}', f'/content/gold_v2/{t}')
    coco['images'].append({'id': img_id, 'file_name': t,
                           'width': 640, 'height': 640})
    for x1, y1, x2, y2, c in filtered[t]:
        coco['annotations'].append({'id': ann_id, 'image_id': img_id,
            'category_id': 1, 'bbox': [x1, y1, x2-x1, y2-y1],
            'area': (x2-x1)*(y2-y1), 'iscrowd': 0})
        ann_id += 1

json.dump(coco, open('/content/gold_v2_coco.json', 'w'))
shutil.make_archive(f'{DRIVE}/gold_v2_tiles', 'zip', '/content', 'gold_v2')
shutil.copy('/content/gold_v2_coco.json', f'{DRIVE}/')
open(f'{DRIVE}/gold_v2_list.txt', 'w').write('\n'.join(sorted(gold)))
print(f'{len(gold)} tiles, {ann_id-1} pre-annotation boxes')
print('Drive: gold_v2_tiles.zip, gold_v2_coco.json, gold_v2_list.txt')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/cotton_bench'
import json

preds = json.load(open(f'{DRIVE}/teacher_test_predictions.json'))
gold = open(f'{DRIVE}/gold_v2_list.txt').read().split()

CONF, MAX_SIDE = 0.30, 120
coco = {'images': [], 'annotations': [],
        'categories': [{'id': 1, 'name': 'cotton_boll'}]}
ann_id = 1
for img_id, t in enumerate(sorted(gold), 1):
    coco['images'].append({'id': img_id, 'file_name': t, 'width': 640, 'height': 640})
    for x1, y1, x2, y2, c in preds[t]:
        if c >= CONF and max(x2 - x1, y2 - y1) <= MAX_SIDE:
            coco['annotations'].append({'id': ann_id, 'image_id': img_id,
                'category_id': 1, 'bbox': [x1, y1, x2 - x1, y2 - y1],
                'area': (x2 - x1) * (y2 - y1), 'iscrowd': 0})
            ann_id += 1
json.dump(coco, open(f'{DRIVE}/gold_v2_coco_c30.json', 'w'))
print(f'{len(gold)} tiles, {ann_id-1} boxes at conf>={CONF} '
      f'({(ann_id-1)/len(gold):.0f} per tile)')